# BERTopic topics in pathology reports and continuous patient risk (post-landmark text)

Split out of [13: pathology terms vs risk](13_pathology_topics_vs_risk.ipynb) on 2026-11-18. It uses the same scores,
landmarks, risk tail groups, post-landmark reports and strata as notebook 13. High and low groups are the highest-risk 10% and lowest-risk 10% at each landmark. Run manuscript notebook 04 first.

1. **BERTopic** on precomputed Clinical ModernBERT report embeddings (read-only), one fit across landmarks.
2. **Topic enrichment**: one-sided Fisher and Mantel-Haenszel tests for each topic, top vs bottom risk tail.
3. **Continuous topic association**: note-level binomial GEE on the raw `risk_score` (patient clusters) and a
   patient-level Spearman sensitivity with patient permutations.

Topic discovery never receives risk scores. The notebook skips with an explanation if the embeddings or the optional
packages (`bertopic`, `umap-learn`, `hdbscan`, `zstandard`; `statsmodels` for the GEE) are unavailable.

Outputs go to `<COMPASS data root>/testing/pathology_topics_vs_risk/`: `pathology_topics.csv`,
`pathology_topic_enrichment.csv`, `pathology_topic_continuous_association.csv`, `pathology_topic_risk_curves.csv` and
`pathology_topic_continuous_coverage.csv`. Only aggregate tables and curves are saved; no patient IDs, raw report text,
representative documents or fitted topic model.

**Timing: post-landmark text only.** At each landmark, a report is used when ADT + landmark < report date ≤
ADT + `WINDOW_MAX_DAYS` (3650 d, the 10-year cap of notebook 11). Days run from ADT start
(`TREATMENT_ANCHOR_DATE` in `DATA_PATH`) to the report's `EVENT_DATE`, dated as in notebook 11. Undated and
pre-landmark reports are dropped. The window is applied per landmark, so a report at ADT + 120 d is used at +0 and +90
but not at +180. Outputs go to the `post_landmark/` subfolder of the output folder.

The topic model is fit once on the post-landmark reports pooled across landmarks (each report once), and the
enrichment and continuous-risk tests at each landmark use that landmark's post-landmark reports. These are
descriptive associations with risk, not prospective validation or causal effects.

The cross-note-type version is [16_clinical_notes_bertopic_vs_risk.ipynb](16_clinical_notes_bertopic_vs_risk.ipynb).


## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
PATHOLOGY_PATH = Path(os.environ.get(
    "PROFILE_PATHOLOGY_NOTES",
    "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/CLINICAL_NOTES/PATHOLOGY_NOTES.parquet"))
OUTPUT_DIR = DATA_ROOT / "text_interpretation"

# Post-landmark text only: at each landmark, notes with ADT + landmark < EVENT_DATE <= ADT + WINDOW_MAX_DAYS.
# Undated notes are dropped. Outputs go to the post_landmark/ subfolder.
WINDOW_MAX_DAYS = 3650             # same 10-year cap as notebook 11
DATA_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"   # ADT anchor dates (TREATMENT_ANCHOR_DATE)
OUTPUT_DIR = OUTPUT_DIR / "post_landmark"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (180,)               # only the ADT +180 d landmark is analysed
RISK_TAIL_FRACTION = 0.10           # highest 10% versus lowest 10% at each landmark

# Pathology term enrichment (top vs bottom risk tail)
TERM_MIN_PATIENTS = 5              # a term must appear in >= this many patients at the landmark
TERM_NGRAMS = (1, 2)               # unigrams and bigrams
TERM_FDR = 0.05                    # Benjamini-Hochberg threshold for the displayed tables
TERM_TOP_N = 25                    # terms shown per landmark x group

# Stratified sensitivity tests (Mantel-Haenszel, van Elteren): report-count band x any non-prostate specimen
ADJUST_REPORT_BANDS = (1, 2, 4)    # band lower edges: 1, 2-3, 4+ reports
SPECIMEN_HEADER_CHARS = 300        # report characters (after PROC_DESC) searched for the specimen site

# Optional BERTopic on the same reports, from precomputed Clinical ModernBERT embeddings (read-only, from the
# clinical_text_embedding_project's knit_embeddings.py). The section skips if packages or files are missing.
CTEP_PROCESSED_DIR = Path(os.environ.get(
    "CTEP_DATA_PATH", "/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/"
)) / "batched_datasets" / "processed_datasets"
EMBED_METADATA_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_metadata.parquet"
EMBED_ARRAY_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_as_array.npy.zst"
TOPIC_MIN_SIZE = 30                # HDBSCAN min_cluster_size, in reports
TOPIC_SEED = 42                    # UMAP random_state

# Continuous topic association: raw score, standardized across eligible patients within each landmark.
CONTINUOUS_MIN_PATIENTS = 30       # minimum independent patient clusters for asymptotic GEE inference
CONTINUOUS_MIN_TOPIC_PATIENTS = 5 # require patients with and without the topic
TOPIC_PERMUTATIONS = 1999         # patient-level Spearman independence null; increase for finer p-values


## Risk scores (all patients)

One full-model XGBoost score per patient and landmark. Invalid scores are excluded. Decile labels support the legacy
contrasts only; coincident cutoffs leave all finite scores in the middle group and do not block continuous analyses.


In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def risk_tail_flags(g):
    if not 0 < RISK_TAIL_FRACTION < 0.5:
        raise ValueError("RISK_TAIL_FRACTION must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    lo, hi = risk[valid].quantile([RISK_TAIL_FRACTION, 1 - RISK_TAIL_FRACTION]) if valid.any() else (np.nan, np.nan)
    if not valid.any() or lo >= hi:
        print(f"Landmark {landmark}: no distinct risk-tail cutoffs; legacy extreme-group contrasts unavailable.")
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    # Only the two risk tails are labelled; everything in between stays blank.
    g["decile_group"] = pd.Series(pd.NA, index=g.index, dtype="string")
    if lo < hi:
        g.loc[risk >= hi, "decile_group"] = "top"
        g.loc[risk <= lo, "decile_group"] = "bottom"
    g.loc[~valid, "decile_group"] = "not scored"
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: risk_tail_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

## Reports, groups and strata

Same loading, risk tail groups and Mantel-Haenszel strata as the term notebook. The definitions cell below
repeats its Fisher / Mantel-Haenszel and BH helpers without running term enrichment.


In [ ]:
import re

import polars as pl

TERM_GROUPS = ("top", "bottom")
CONTROL_GROUP = "middle"   # patients in neither decile: a label only, never analysed
# (group, control): one-sided tests for terms / topics overrepresented in group relative to control.
CONTRASTS = (("top", "bottom"), ("bottom", "top"))   # one-sided, so both directions

# --- Specimen site (for the stratified sensitivity test): the site named first in PROC_DESC + the report's opening.
SPECIMEN_SITES = (
    ("prostate", re.compile(r"\b(?:prostat\w*|turp|transurethral|seminal\s+vesicles?)\b")),
    ("non-prostate", re.compile(r"\b(?:bone|marrow|liver|hepatic|lymph|nodes?|nodal|lung|pulmonary|pleura\w*|adrenal"
                                r"|brain|dura\w*|spine|spinal|vertebra\w*|soft\s+tissue|peritone\w*|omentum|omental)\b")),
)


def specimen_site(header):
    """'prostate', 'non-prostate' or 'unspecified': whichever site is named first in a lowercased header."""
    hits = [(m.start(), site) for site, rx in SPECIMEN_SITES for m in [rx.search(header)] if m]
    return min(hits)[1] if hits else "unspecified"


# Per-landmark group for every scored patient; a blank decile group means scored but in neither decile.
groups_by_lm = {}
for lm, g in by_lm.items():
    ids = pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64"), name="DFCI_MRN")
    grp = pd.Series(g["decile_group"].fillna(CONTROL_GROUP).to_numpy(), index=ids, name="group")
    groups_by_lm[lm] = grp[grp.ne("not scored")]

scored_ids = sorted({int(i) for grp in groups_by_lm.values() for i in grp.index.dropna()})
# RPT_ID (when present) links reports to the optional embeddings; PROC_DESC (when present) helps type the specimen.
note_cols = ["DFCI_MRN", "EVENT_DATE", "RPT_TEXT"]
schema = pl.scan_parquet(PATHOLOGY_PATH).collect_schema().names()
note_cols += [c for c in ("RPT_ID", "PROC_DESC") if c in schema]
all_notes = (pl.scan_parquet(PATHOLOGY_PATH)
             .select(note_cols)
             .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids))
             .collect().to_pandas())
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes["note_row"] = np.arange(len(all_notes))      # row position, used to look up per-report lexicon hits

# Specimen site, once per report.
header = all_notes["RPT_TEXT"].str[:SPECIMEN_HEADER_CHARS]
if "PROC_DESC" in all_notes:
    header = all_notes["PROC_DESC"].fillna("").astype(str) + " " + header
all_notes["specimen_site"] = header.str.lower().map(specimen_site) if len(all_notes) else pd.Series(dtype=object)


# --- Post-landmark window. Days from ADT start = EVENT_DATE (its YYYY-MM-DD prefix) - TREATMENT_ANCHOR_DATE,
# dated as in notebook 11. Undated notes are dropped.
anchor_cols = ["DFCI_MRN", "TREATMENT_ANCHOR_DATE"]
anchor = (pd.read_parquet(DATA_PATH, columns=anchor_cols) if str(DATA_PATH).endswith((".parquet", ".pq"))
          else pd.read_csv(DATA_PATH, usecols=anchor_cols, low_memory=False)).dropna().drop_duplicates()
anchor["TREATMENT_ANCHOR_DATE"] = pd.to_datetime(anchor["TREATMENT_ANCHOR_DATE"], errors="coerce", format="mixed")
anchor = anchor.dropna().drop_duplicates()
anchor["DFCI_MRN"] = pd.to_numeric(anchor["DFCI_MRN"], errors="raise").astype("int64")
if anchor["DFCI_MRN"].duplicated().any():
    raise ValueError("More than one TREATMENT_ANCHOR_DATE for some patients.")
anchor = anchor.set_index("DFCI_MRN")["TREATMENT_ANCHOR_DATE"]
n_missing = sum(i not in anchor.index for i in scored_ids)
if n_missing:
    raise ValueError(f"{n_missing} scored patients lack an ADT anchor in DATA_PATH; reconcile it with the scores.")
note_date = pd.to_datetime(all_notes["EVENT_DATE"].astype("string").str.extract(r"^(\d{4}-\d{2}-\d{2})", expand=False),
                           errors="coerce")
all_notes["days_from_adt"] = (note_date - all_notes["DFCI_MRN"].map(anchor)).dt.days.astype(float).to_numpy()


def in_window(reports, lm):
    """Rows of `reports` after ADT + `lm` and at or before ADT + WINDOW_MAX_DAYS (undated rows never)."""
    days = reports["days_from_adt"]
    return (days.gt(lm) & days.le(WINDOW_MAX_DAYS)).fillna(False).to_numpy(dtype=bool)


def window_summary(rows):
    """Prints, per landmark, how many of the scored patients' notes are post-landmark (aggregate counts only)."""
    print(f"Post-landmark notes (ADT + landmark < note date <= ADT + {WINDOW_MAX_DAYS} d; undated notes dropped):")
    for r in rows:
        print(f"  landmark {r['landmark']}: {r['in_window']:,} of {r['any_date']:,} notes "
              f"({r['undated']:,} undated), {r['patients_in_window']:,} patients")


# The post-landmark reports of the patients scored at each landmark.
# Each patient's stratum: report count band (ADJUST_REPORT_BANDS) x any non-prostate-site report, within the window.
reports_by_lm, strata_by_lm, size_rows, window_rows = {}, {}, [], []
report_cols = ["DFCI_MRN", "RPT_TEXT", "specimen_site", "note_row"] + note_cols[3:]
for lm, grp in groups_by_lm.items():
    pool = all_notes[all_notes["DFCI_MRN"].isin(grp.index)]
    rep_lm = pool.loc[in_window(pool, lm), report_cols].reset_index(drop=True)
    reports_by_lm[lm] = rep_lm
    window_rows.append(dict(landmark=lm, any_date=len(pool), in_window=len(rep_lm),
                            undated=int(pool["days_from_adt"].isna().sum()),
                            patients_in_window=int(rep_lm["DFCI_MRN"].nunique())))
    per = (rep_lm.assign(non_prostate=rep_lm["specimen_site"].eq("non-prostate"))
           .groupby("DFCI_MRN").agg(reports=("non_prostate", "size"), non_prostate=("non_prostate", "any")))
    band = np.searchsorted(ADJUST_REPORT_BANDS, per["reports"].to_numpy(), side="right") - 1
    strata_by_lm[lm] = pd.Series(band * 2 + per["non_prostate"].to_numpy(dtype=int), index=per.index, name="stratum")
    has_report = grp.index.isin(rep_lm["DFCI_MRN"])
    for name in TERM_GROUPS:
        in_group = grp.eq(name).to_numpy()
        members = per.index.isin(grp.index[in_group])
        size_rows.append({"landmark": lm, "group": name, "patients_scored": int(in_group.sum()),
                          "patients_with_report": int((in_group & has_report).sum()),
                          "reports": int(rep_lm["DFCI_MRN"].isin(grp.index[in_group]).sum()),
                          "median_reports": float(per.loc[members, "reports"].median()) if members.any() else np.nan,
                          "patients_with_non_prostate_report": int(per.loc[members, "non_prostate"].sum())})

sites = all_notes["specimen_site"].value_counts()
print(f"Specimen site of the {len(all_notes):,} pathology reports of scored patients (all dates): "
      + ", ".join(f"{site} {int(sites.get(site, 0)):,}" for site in ("prostate", "non-prostate", "unspecified")))
window_summary(window_rows)
group_sizes = pd.DataFrame(size_rows).set_index(["landmark", "group"])
group_sizes


In [ ]:
try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

from scipy import sparse
from scipy.stats import hypergeom, norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# Applied after lowercasing: letters only, >= 3 characters.
TOKEN_PATTERN = r"(?u)\b[a-z][a-z]{2,}\b"
MH_COLS = ["odds_ratio_mh", "p_value_mh", "q_value_mh"]
ENRICHMENT_COLS = ["landmark", "group", "control", "term", "n_group", "n_control", "group_with_term", "control_with_term",
                   "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS,
                   "mean_tfidf_group", "mean_tfidf_control"]


def bh_qvalues(p):
    """Benjamini-Hochberg q-values over the finite p values; NaN stays NaN (cluster scipy predates
    false_discovery_control)."""
    p = np.asarray(p, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if ok.size:
        order = ok[np.argsort(p[ok])]
        ranked = p[order] * ok.size / np.arange(1, ok.size + 1)
        q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def patient_term_matrices(reports, ids):
    """Patient x term presence and TF-IDF, with terms counted within each report and summed per patient.
    """
    text = reports["RPT_TEXT"].to_numpy()
    vec = CountVectorizer(lowercase=True, stop_words="english", token_pattern=TOKEN_PATTERN,
                          ngram_range=TERM_NGRAMS, min_df=TERM_MIN_PATIENTS)
    try:
        # Report-level min_df is a safe prefilter: a term in >= k patients is in >= k reports.
        counts = vec.fit_transform(text)
    except ValueError:  # no term survives (or fewer reports than TERM_MIN_PATIENTS)
        return np.array([], dtype=object), None, None
    rows = ids.get_indexer(reports["DFCI_MRN"])
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))),
                                   shape=(len(ids), len(rows)))
    patient_counts = (to_patient @ counts).tocsc()
    present = (patient_counts > 0).astype(np.int64)
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return np.array([], dtype=object), None, None
    patient_counts, present = patient_counts[:, keep], present[:, keep]
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(patient_counts)
    return vec.get_feature_names_out()[keep], present.tocsr(), tfidf.tocsr()


def colsum(m):
    return np.asarray(m.sum(axis=0)).ravel().astype(float)


def mantel_haenszel(present, in_group, control, strata):
    """One-sided, continuity-corrected Mantel-Haenszel test and MH odds ratio per feature column, across strata.

    A stratum without patients from both sides carries no information and is skipped; if none is left, both are NaN.
    The odds ratio adds 0.5 to every cell of a stratum that has an empty cell.
    """
    n_feat = present.shape[1]
    A, E, V, R, S = (np.zeros(n_feat) for _ in range(5))
    informative = False
    for k in pd.unique(strata[in_group | control]):
        g, c = in_group & (strata == k), control & (strata == k)
        n1, n0 = int(g.sum()), int(c.sum())
        if not (n1 and n0):
            continue
        informative = True
        n = n1 + n0
        a, cc = colsum(present[g]), colsum(present[c])
        b, d, m = n1 - a, n0 - cc, a + cc
        A += a
        E += n1 * m / n
        V += n1 * n0 * m * (n - m) / (n ** 2 * (n - 1))
        h = 0.5 * ((a == 0) | (b == 0) | (cc == 0) | (d == 0))
        R += (a + h) * (d + h) / (n + 4 * h)
        S += (b + h) * (cc + h) / (n + 4 * h)
    if not informative:
        return np.full(n_feat, np.nan), np.full(n_feat, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(V > 0, norm.sf((A - E - 0.5) / np.sqrt(V)), 1.0)
    return R / S, p


def fisher_enrichment(present, label, features, landmark, strata=None):
    """One-sided Fisher overrepresentation of each feature column for each (group, control) in CONTRASTS.

    present: patients x features 0/1 sparse matrix; label: each patient's group; strata (optional): each patient's
    stratum for the Mantel-Haenszel columns (NaN without it). Rows come out in CONTRASTS order, len(features) rows
    per contrast whose group and control both have patients.
    """
    out = []
    for name, ref in CONTRASTS:
        in_group, control = label == name, label == ref
        n1, n0 = int(in_group.sum()), int(control.sum())
        if n1 == 0 or n0 == 0:
            continue
        a, c = colsum(present[in_group]), colsum(present[control])
        p = hypergeom.sf(a - 1, n1 + n0, a + c, n1)       # P(X >= a): one-sided Fisher
        if strata is None:
            or_mh = p_mh = np.full(len(features), np.nan)
        else:
            or_mh, p_mh = mantel_haenszel(present, in_group, control, np.asarray(strata))
        out.append(pd.DataFrame({
            "landmark": landmark, "group": name, "control": ref, "feature": features, "n_group": n1, "n_control": n0,
            "group_with_feature": a.astype(int), "control_with_feature": c.astype(int),
            "pct_group": a / n1, "pct_control": c / n0,
            "odds_ratio": ((a + 0.5) * (n0 - c + 0.5)) / ((n1 - a + 0.5) * (c + 0.5)),
            "p_value": p, "q_value": bh_qvalues(p),
            "odds_ratio_mh": or_mh, "p_value_mh": p_mh, "q_value_mh": bh_qvalues(p_mh),
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def sort_contrasts(df):
    """Landmark, then CONTRASTS order, then p value and odds ratio."""
    rank = {pair: i for i, pair in enumerate(CONTRASTS)}
    order = pd.Series([rank[pair] for pair in zip(df["group"], df["control"])], index=df.index, dtype=float)
    return (df.assign(_contrast=order)
            .sort_values(["landmark", "_contrast", "p_value", "odds_ratio"], ascending=[True, True, True, False])
            .drop(columns="_contrast").reset_index(drop=True))


def contrast_title(lm, name, ref):
    return f"Landmark {lm} · {name} {RISK_TAIL_FRACTION:.0%} vs {ref} {RISK_TAIL_FRACTION:.0%}"


## BERTopic topics

Groups and post-landmark reports are the same as in the term notebook. Instead of single terms, reports are clustered into topics,
and topics are tested for overrepresentation in the same two contrasts (top vs bottom and bottom vs top).

- **Embeddings:** precomputed, mean-pooled Clinical ModernBERT embeddings of each pathology report, read (never written)
  from the clinical_text_embedding_project (`full_clinical_notes_embeddings_as_array.npy.zst` and its metadata parquet),
  joined on `DFCI_MRN` + `RPT_ID` with one-to-one embedding validation. No language model is loaded and nothing is downloaded. Reports outside that project's cohort have
  no embedding; the coverage table shows how many patients each contrast keeps.
- **Model:** one BERTopic fit on the union of embedded post-landmark reports across landmarks (each report once),
  so topic numbers mean the same thing at every landmark. UMAP (5 dimensions, cosine, `TOPIC_SEED`) then HDBSCAN
  (`TOPIC_MIN_SIZE` reports). Topic words use only the terms that appear in at least `TERM_MIN_PATIENTS` patients.
- **Test:** a patient "has" a topic if any of their embedded post-landmark reports at that landmark is assigned to it. Outlier reports
  (topic -1) are not a feature, but their patients stay in the denominators. Denominators are patients with at least one
  embedded report. Topics in fewer than `TERM_MIN_PATIENTS` patients at a landmark are dropped there.
  Same one-sided Fisher test, BH q-values and odds ratio as the terms.
- **PHI:** BERTopic keeps representative documents (raw report text) inside the fitted model. They are dropped from
  every table and the model is never saved; only topic sizes and top words are written.

The section needs `bertopic`, `umap-learn`, `hdbscan` and `zstandard` (commented out in `requirements.txt`). Without
them, or without the embedding files, it prints why it skipped and writes nothing; the term notebook is unaffected.
Loading the embeddings needs only `zstandard`.

In [ ]:
import importlib
import json
import io

TOPIC_PACKAGES = {"zstandard": "zstandard", "bertopic": "bertopic", "umap": "umap-learn", "hdbscan": "hdbscan"}


def optional_import(name):
    try:
        return importlib.import_module(name)
    except Exception:   # ImportError, or a broken install (e.g. numba) that fails on import
        return None


def rpt_key(values):
    """RPT_ID as a string join key; whole-number IDs lose any float formatting ("12.0" -> "12")."""
    s = pd.Series(values).reset_index(drop=True)
    num = pd.to_numeric(s, errors="coerce")
    whole = num.notna() & np.isfinite(num) & (num % 1 == 0)
    key = s.astype("string").str.strip()
    key[whole] = num[whole].astype("int64").astype(str)
    return key.where(s.notna())


def pathology_keys(frame):
    """Pathology identity includes the patient; report IDs need not be globally unique."""
    patient, report = rpt_key(frame["DFCI_MRN"]), rpt_key(frame["RPT_ID"])
    valid = patient.notna() & report.notna()
    keys = pd.Series(pd.NA, index=range(len(frame)), dtype="string")
    keys.loc[valid] = [json.dumps([p, r], separators=(",", ":"))
                       for p, r in zip(patient[valid], report[valid])]
    return keys


def _read_exact(fh, n):
    parts, got = [], 0
    while got < n:
        chunk = fh.read(n - got)
        if not chunk:
            raise ValueError("embedding array ended before its header's shape")
        parts.append(chunk)
        got += len(chunk)
    return b"".join(parts)


def read_embedding_rows(path, rows, zstd, chunk_rows=16384):
    """Rows `rows` (sorted, unique) of a zstd-compressed 2-D .npy array, as float32, in one streaming pass.

    The full array (every note type for the embedding cohort) is never held in memory: it is decompressed in
    chunk_rows blocks and only the requested rows are kept.
    """
    rows = np.asarray(rows, dtype=np.int64)
    with open(path, "rb") as fh, zstd.ZstdDecompressor().stream_reader(fh) as raw:
        version = np.lib.format.read_magic(raw)
        readers = {(1, 0): np.lib.format.read_array_header_1_0, (2, 0): np.lib.format.read_array_header_2_0}
        if version not in readers:
            raise ValueError(f"unsupported .npy version {version}")
        shape, fortran_order, dtype = readers[version](raw)
        if fortran_order or len(shape) != 2 or dtype.hasobject:
            raise ValueError(f"expected a C-order 2-D numeric array, got shape {shape}")
        n_rows, dim = shape
        if rows.size and (rows[0] < 0 or rows[-1] >= n_rows):
            raise ValueError(f"EMBEDDING_INDEX outside the array's {n_rows:,} rows")
        out = np.empty((rows.size, dim), dtype=np.float32)
        start, stop = 0, int(rows[-1]) + 1 if rows.size else 0
        while start < stop:
            k = min(chunk_rows, stop - start)
            block = np.frombuffer(_read_exact(raw, k * dim * dtype.itemsize), dtype=dtype).reshape(k, dim)
            lo, hi = np.searchsorted(rows, [start, start + k])
            out[lo:hi] = block[rows[lo:hi] - start]
            start += k
    return out


topic_mods = {name: optional_import(name) for name in TOPIC_PACKAGES}
missing = [pip for name, pip in TOPIC_PACKAGES.items() if topic_mods[name] is None]

# Loading the embeddings needs only zstandard.
embed_skip = None
if topic_mods["zstandard"] is None:
    embed_skip = "packages not installed: zstandard"
elif not (EMBED_METADATA_PATH.exists() and EMBED_ARRAY_PATH.exists()):
    embed_skip = f"precomputed embeddings not found under {EMBED_ARRAY_PATH.parent}"
elif "RPT_ID" not in all_notes:
    embed_skip = "pathology notes have no RPT_ID to join the embeddings on"

embed_docs = report_embeddings = None   # embed_docs row i <-> report_embeddings row i
if embed_skip is None:
    # Each post-landmark report once, across landmarks (a report can be post-landmark at several landmarks).
    embed_docs = (pd.concat(reports_by_lm.values(), ignore_index=True)
                  .assign(rpt_key=lambda d: pathology_keys(d).to_numpy())
                  .dropna(subset=["rpt_key"]).drop_duplicates("rpt_key", ignore_index=True))
    meta = (pl.scan_parquet(EMBED_METADATA_PATH)
            .filter(pl.col("NOTE_TYPE") == "Pathology")
            .select("DFCI_MRN", "RPT_ID", "EMBEDDING_INDEX")
            .collect().to_pandas())
    meta["rpt_key"] = pathology_keys(meta).to_numpy()
    meta = meta.dropna(subset=["rpt_key", "EMBEDDING_INDEX"])
    meta = meta[meta["rpt_key"].isin(embed_docs["rpt_key"])][["rpt_key", "EMBEDDING_INDEX"]].drop_duplicates()
    index = pd.to_numeric(meta["EMBEDDING_INDEX"], errors="coerce")
    if not (np.isfinite(index) & index.ge(0) & index.mod(1).eq(0)).all():
        raise ValueError("Embedding indices must be finite nonnegative integers.")
    meta["EMBEDDING_INDEX"] = index.astype("int64")
    if meta["rpt_key"].duplicated().any() or meta["EMBEDDING_INDEX"].duplicated().any():
        raise ValueError("Ambiguous embedding metadata: note identities and embedding rows must be one-to-one.")
    n_window = len(embed_docs)
    embed_docs = (embed_docs.merge(meta[["rpt_key", "EMBEDDING_INDEX"]], on="rpt_key", how="inner",
                                   validate="one_to_one")
                  .sort_values("EMBEDDING_INDEX", ignore_index=True))
    print(f"{len(embed_docs):,} of {n_window:,} pathology reports have a precomputed embedding")
    if embed_docs.empty:
        embed_skip, embed_docs = "no pathology report has a precomputed embedding", None
    else:
        report_embeddings = read_embedding_rows(EMBED_ARRAY_PATH, embed_docs["EMBEDDING_INDEX"].astype("int64").to_numpy(),
                                                topic_mods["zstandard"])

topic_skip = None
if missing:
    topic_skip = f"packages not installed: {', '.join(missing)}"
elif embed_skip is not None:
    topic_skip = embed_skip
elif len(embed_docs) <= 2 * TOPIC_MIN_SIZE:
    topic_skip = f"only {len(embed_docs):,} embedded reports (need more than {2 * TOPIC_MIN_SIZE})"
else:
    # BERTopic's English preprocessing deletes every character outside [A-Za-z0-9 ] before its class TF-IDF
    # ("lymph-node" -> "lymphnode") but scores representative documents on the text as passed, so a vocabulary
    # term seen only beside punctuation gets an infinite IDF and the fit fails. Replacing those characters with
    # spaces first gives both passes the same text (BERTopic's cleaning is then a no-op) and keeps the tokens.
    topic_text = (embed_docs["RPT_TEXT"].astype("string").fillna("")
                  .str.replace(r"[^A-Za-z0-9 ]+", " ", regex=True).to_numpy(dtype=object))
    # Topic words come only from terms in >= TERM_MIN_PATIENTS patients, the same rule as the term enrichment.
    vocab, _, _ = patient_term_matrices(embed_docs.assign(RPT_TEXT=topic_text), pd.Index(sorted(embed_docs["DFCI_MRN"].unique()), dtype="Int64"))
    if not len(vocab):
        topic_skip = f"no term appears in >= {TERM_MIN_PATIENTS} patients' embedded reports"

topic_docs = None
if topic_skip is None:
    topic_model = topic_mods["bertopic"].BERTopic(
        embedding_model=None,   # embeddings are passed in: no backend is selected or downloaded
        umap_model=topic_mods["umap"].UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                                           random_state=TOPIC_SEED),
        hdbscan_model=topic_mods["hdbscan"].HDBSCAN(min_cluster_size=TOPIC_MIN_SIZE, metric="euclidean",
                                                    cluster_selection_method="eom", prediction_data=False),
        vectorizer_model=CountVectorizer(vocabulary=list(vocab), ngram_range=TERM_NGRAMS, lowercase=True,
                                         stop_words="english", token_pattern=TOKEN_PATTERN),   # = term enrichment
        calculate_probabilities=False,
        verbose=False,
    )
    assigned, _ = topic_model.fit_transform(list(topic_text), embeddings=report_embeddings)
    topic_docs = embed_docs.assign(topic=np.asarray(assigned, dtype=int))
    info = topic_model.get_topic_info()
    words = {int(t): " ".join(w for w, _ in (topic_model.get_topic(t) or [])) for t in info["Topic"]}
    del topic_model, info   # the model holds representative report text: never saved or shown
    topic_table = (topic_docs.groupby("topic")
                   .agg(n_reports=("rpt_key", "size"), n_patients=("DFCI_MRN", "nunique"))
                   .reset_index())
    topic_table["top_words"] = topic_table["topic"].map(words).fillna("")
    topic_table.loc[topic_table["topic"].eq(-1), "top_words"] = "(outliers: no topic)"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    topics_path = OUTPUT_DIR / "pathology_topics.csv"
    topic_table.to_csv(topics_path, index=False)
    print(f"wrote {topics_path} ({topic_table['topic'].ge(0).sum():,} topics, "
          f"{topic_table.loc[topic_table['topic'].eq(-1), 'n_reports'].sum():,} outlier reports)")
else:
    print(f"BERTopic section skipped ({topic_skip}); no topic files written.")


In [ ]:
TOPIC_COLS = ["landmark", "group", "control", "topic", "top_words", "n_group", "n_control", "group_with_topic",
              "control_with_topic", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value", *MH_COLS]


def topic_enrichment(docs, groups, landmark):
    """Fisher overrepresentation of each topic for each contrast in CONTRASTS at one landmark."""
    ids = pd.Index(sorted(docs["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 embedded report
    topics = np.array(sorted(t for t in docs["topic"].unique() if t >= 0))
    hits = docs[docs["topic"].ge(0)]
    present = sparse.csr_matrix(
        (np.ones(len(hits)), (ids.get_indexer(hits["DFCI_MRN"]), np.searchsorted(topics, hits["topic"]))),
        shape=(len(ids), len(topics)))
    present = (present > 0).astype(np.int64).tocsc()
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = fisher_enrichment(present[:, keep].tocsr(), groups.reindex(ids).to_numpy(), topics[keep], landmark,
                            strata_by_lm[landmark].reindex(ids).to_numpy())
    if res.empty:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = res.rename(columns={"feature": "topic", "group_with_feature": "group_with_topic",
                              "control_with_feature": "control_with_topic"})
    return res.assign(top_words=res["topic"].map(topic_table.set_index("topic")["top_words"]))


if topic_docs is None:
    print("BERTopic section skipped; see the previous cell.")
else:
    topic_frames, coverage_rows = [], []
    keys = topic_docs.set_index("rpt_key")
    for lm in LANDMARKS:
        if lm not in reports_by_lm:
            continue
        grp = groups_by_lm[lm]
        lm_keys = pathology_keys(reports_by_lm[lm]).dropna().drop_duplicates()
        docs_lm = keys.loc[keys.index.intersection(pd.Index(lm_keys)), ["DFCI_MRN", "topic"]].reset_index()
        topic_frames.append(topic_enrichment(docs_lm, grp, lm))
        for name in TERM_GROUPS:
            members = grp.index[grp.eq(name).to_numpy()]
            in_group = docs_lm["DFCI_MRN"].isin(members)
            coverage_rows.append({"landmark": lm, "group": name,
                                  "patients_with_report": group_sizes.loc[(lm, name), "patients_with_report"],
                                  "patients_with_embedded_report": int(docs_lm.loc[in_group, "DFCI_MRN"].nunique()),
                                  "embedded_reports": int(in_group.sum()),
                                  "outlier_reports": int((in_group & docs_lm["topic"].eq(-1)).sum())})
    topic_enrich = sort_contrasts(pd.concat(topic_frames, ignore_index=True).reindex(columns=TOPIC_COLS))
    topic_enrich_path = OUTPUT_DIR / "pathology_topic_enrichment.csv"
    topic_enrich.to_csv(topic_enrich_path, index=False)
    print(f"wrote {topic_enrich_path} ({len(topic_enrich):,} topic x contrast rows)")

    topic_coverage = pd.DataFrame(coverage_rows).set_index(["landmark", "group"])
    display(topic_coverage)
    topic_show = ["topic", "top_words", "group_with_topic", "n_group", "pct_group", "control_with_topic",
                  "n_control", "pct_control", "odds_ratio", "q_value", "odds_ratio_mh", "q_value_mh"]
    for lm in LANDMARKS:
        for name, ref in CONTRASTS:
            sub = topic_enrich[(topic_enrich["landmark"] == lm) & (topic_enrich["group"] == name)
                               & (topic_enrich["control"] == ref)]
            title = f"{contrast_title(lm, name, ref)} (topics)"
            if sub.empty:
                print(f"\n{title}: no topics (too few patients or reports)")
                continue
            print(f"\n{title} (q < {TERM_FDR}: {int((sub['q_value'] < TERM_FDR).sum()):,} topics)")
            display(sub[topic_show].head(TERM_TOP_N).round(3).reset_index(drop=True))

## Continuous topic association: does topic frequency change with patient risk?

Yes: use every finite patient score and keep `risk_score` continuous. The models below fit **each topic vs all other
notes** separately at each landmark; topic -1 is included among other notes and in patient note-count denominators.
Only patients with an embedded pathology report are represented. Missing embeddings are not negative topic assignments;
the coverage table reports exclusions. The score is a model output, not a calibrated event probability.

- **Note-level binomial GEE (primary):** `I(note topic == k) ~ standardized raw risk score`, clustered by patient,
  with an independence working correlation and robust sandwich standard errors. The odds ratio is per one patient-level
  SD of risk, computed among eligible patients at that landmark. OR > 1 means the topic becomes more common in notes
  as risk increases. The adjusted sensitivity adds log(1 + embedded report count) and report specimen site (non-prostate
  and unspecified indicators). Constant covariates are omitted; collinear or unstable fits are marked as skipped.
  GEE handles correlated notes, but this estimates a **note-weighted** association: patients with more notes contribute more.
- **Patient-level sensitivity:** fraction of a patient's embedded notes assigned to each topic, correlated with the raw
  risk score using Spearman rho. Each patient contributes once. Two-sided permutation p-values shuffle whole patient scores
  under an independence null, preserving ties and each patient's full note collection. This is unadjusted and detects
  monotonic associations; it does not assume the score is linearly spaced. Set `TOPIC_PERMUTATIONS` higher for finer p-values.
- **Inference:** require `CONTINUOUS_MIN_PATIENTS` independent patient clusters and at least
  `CONTINUOUS_MIN_TOPIC_PATIENTS` patients contributing topic notes and other notes (these sets may overlap).
  Constant scores/fractions and failed fits are explicitly skipped. The patient minimum is a screening rule, not a guarantee
  of reliable asymptotic intervals. BH correction is across **all topics and landmarks within each method**; methods are
  separate analysis families. Skipped fits have no p/q-value. Topic discovery uncertainty is not included.
- **Curves:** unadjusted GEE fitted probabilities over the observed raw score range, with pointwise 95% robust intervals.
  There are no risk bins. These curves assume a linear effect on log odds and are not evidence of a nonlinear relationship.
  A spline of risk in the same clustered model is a useful later extension if the cohort supports the extra parameters.

Only post-landmark reports enter these fits, so associations can reflect later progression, sampling intensity, or
treatment, not information available at prediction time. Report-count and specimen adjustment are descriptive
sensitivities, not a causal correction. Scores are standardized separately per landmark;
ORs therefore refer to different score SDs and should not be treated as independent replication across landmarks.

Methods: [statsmodels GEE documentation](https://www.statsmodels.org/stable/gee.html),
[SciPy Spearman guidance](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html),
[paired permutation tests](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html).
Install `statsmodels>=0.14` for GEE; patient-level correlations still run without it.


In [ ]:
import warnings

from scipy.special import expit
from scipy.stats import permutation_test, rankdata

CONTINUOUS_COLS = [
    "landmark", "topic", "top_words", "model", "n_patients", "n_reports", "n_topic_reports",
    "n_patients_with_topic", "n_patients_with_other_reports", "risk_mean", "risk_sd",
    "beta_per_sd", "odds_ratio_per_sd", "ci95_low", "ci95_high", "spearman_rho",
    "p_value", "q_value", "status",
]
CURVE_COLS = ["landmark", "topic", "top_words", "risk_score", "probability", "ci95_low", "ci95_high"]


def continuous_topic_association(docs, risk, landmark, top_words):
    """One note per row; risk is a unique patient-indexed Series of raw scores.

    GEE targets note-level topic odds; Spearman targets equally weighted patient topic fractions.
    Outlier notes (-1) remain in denominators. Return aggregate effects and unadjusted fitted curves only.
    """
    if risk.index.has_duplicates:
        raise ValueError("Expected one risk score per patient at this landmark.")
    if TOPIC_PERMUTATIONS < 1:
        raise ValueError("TOPIC_PERMUTATIONS must be positive.")
    docs = docs.copy()
    docs["risk_score"] = pd.to_numeric(docs["DFCI_MRN"].map(risk), errors="coerce")
    docs = docs[np.isfinite(docs["risk_score"])].reset_index(drop=True)
    patients = docs.groupby("DFCI_MRN").agg(risk_score=("risk_score", "first"), n_reports=("topic", "size"))
    n_patients = len(patients)
    mean, sd = patients["risk_score"].mean(), patients["risk_score"].std(ddof=0)
    topics = sorted(t for t in docs["topic"].unique() if t >= 0)
    sm = optional_import("statsmodels.api")
    results, curves = [], []
    for topic in topics:
        y = docs["topic"].eq(topic).astype(float)
        counts = y.groupby(docs["DFCI_MRN"]).sum().reindex(patients.index, fill_value=0)
        fraction = counts / patients["n_reports"]
        base = dict(landmark=landmark, topic=topic, top_words=top_words.get(topic, ""),
                    n_patients=n_patients, n_reports=len(docs), n_topic_reports=int(y.sum()),
                    n_patients_with_topic=int(counts.gt(0).sum()),
                    n_patients_with_other_reports=int(counts.lt(patients["n_reports"]).sum()),
                    risk_mean=mean, risk_sd=sd)
        reason = None
        if n_patients < CONTINUOUS_MIN_PATIENTS:
            reason = "too few patient clusters"
        elif not np.isfinite(sd) or sd <= 0:
            reason = "constant risk score"
        elif min(base["n_patients_with_topic"], base["n_patients_with_other_reports"]) < CONTINUOUS_MIN_TOPIC_PATIENTS:
            reason = "too few patients with topic or other notes"
        for model in ("gee", "gee_adjusted", "patient_fraction_spearman"):
            row = dict(base, model=model, status="ok")
            if reason:
                row["status"] = "skipped: " + reason
            elif model == "patient_fraction_spearman":
                if fraction.nunique() < 2:
                    row["status"] = "skipped: constant patient topic fraction"
                else:
                    # Rank once; permute whole patients, never individual notes. Two-sided independence test.
                    x = rankdata(patients["risk_score"].to_numpy())
                    x = (x - x.mean()) / np.linalg.norm(x - x.mean())
                    f = rankdata(fraction.to_numpy())
                    f = (f - f.mean()) / np.linalg.norm(f - f.mean())
                    def statistic(permuted, axis=-1):
                        return np.sum(permuted * f, axis=axis)
                    test = permutation_test((x,), statistic, permutation_type="pairings", vectorized=True,
                                            n_resamples=TOPIC_PERMUTATIONS, batch=128, alternative="two-sided",
                                            random_state=TOPIC_SEED)
                    row.update(spearman_rho=float(test.statistic), p_value=float(test.pvalue))
            elif sm is None:
                row["status"] = "skipped: statsmodels not installed or unavailable"
            else:
                x = pd.DataFrame({"intercept": np.ones(len(docs)), "risk_z": (docs["risk_score"] - mean) / sd})
                if model == "gee_adjusted":
                    covariates = {
                        "log_report_count": np.log1p(docs["DFCI_MRN"].map(patients["n_reports"]).to_numpy()),
                        "non_prostate": docs["specimen_site"].eq("non-prostate").astype(float).to_numpy(),
                        "unspecified_site": docs["specimen_site"].eq("unspecified").astype(float).to_numpy(),
                    }
                    for name, values in covariates.items():
                        if np.ptp(values) > 0:
                            x[name] = values
                if np.linalg.matrix_rank(x.to_numpy()) < x.shape[1]:
                    row["status"] = "skipped: collinear design"
                else:
                    try:
                        with warnings.catch_warnings(record=True) as caught:
                            warnings.simplefilter("always")
                            fit = sm.GEE(y, x, groups=docs["DFCI_MRN"].to_numpy(dtype=np.int64),
                                         family=sm.families.Binomial(), cov_struct=sm.cov_struct.Independence()
                                         ).fit(maxiter=100, cov_type="robust")
                        beta, se = float(fit.params["risk_z"]), float(fit.bse["risk_z"])
                        if caught or not fit.converged or not np.isfinite([beta, se]).all() or se <= 0:
                            row["status"] = "skipped: fit warning, nonconvergence, or invalid covariance"
                        elif max(abs(beta - 1.96 * se), abs(beta + 1.96 * se)) > 700:
                            row["status"] = "skipped: unstable effect estimate"
                        else:
                            row.update(beta_per_sd=beta, odds_ratio_per_sd=np.exp(beta),
                                       ci95_low=np.exp(beta - 1.96 * se), ci95_high=np.exp(beta + 1.96 * se),
                                       p_value=float(fit.pvalues["risk_z"]))
                            if model == "gee":
                                grid = np.linspace(patients["risk_score"].min(), patients["risk_score"].max(), 100)
                                design = np.column_stack([np.ones(len(grid)), (grid - mean) / sd])
                                eta = design @ fit.params.to_numpy()
                                eta_se = np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", design, fit.cov_params(), design), 0))
                                curves.append(pd.DataFrame(dict(landmark=landmark, topic=topic,
                                    top_words=top_words.get(topic, ""), risk_score=grid, probability=expit(eta),
                                    ci95_low=expit(eta - 1.96 * eta_se), ci95_high=expit(eta + 1.96 * eta_se))))
                    except (ValueError, np.linalg.LinAlgError, FloatingPointError, OverflowError) as exc:
                        row["status"] = f"skipped: model fit failed ({type(exc).__name__})"
            results.append(row)
    effects = pd.DataFrame(results).reindex(columns=CONTINUOUS_COLS)
    curves = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame(columns=CURVE_COLS)
    return effects, curves


continuous_frames, curve_frames, continuous_coverage = [], [], []
if topic_docs is None:
    print("Continuous topic analysis skipped: BERTopic assignments unavailable.")
else:
    words = topic_table.set_index("topic")["top_words"].to_dict()
    for lm in LANDMARKS:
        g = by_lm[lm]
        risk = pd.Series(g["risk_score"].to_numpy(dtype=float),
                         index=pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64")))
        # Same report membership as the legacy analysis; all patients, regardless of decile label.
        lm_keys = pathology_keys(reports_by_lm[lm]).dropna().unique()
        docs = topic_docs[topic_docs["rpt_key"].isin(lm_keys)].copy()
        finite_ids = risk.index[np.isfinite(risk)]
        docs = docs[docs["DFCI_MRN"].isin(finite_ids)]
        continuous_coverage.append(dict(landmark=lm, patients_scored=len(finite_ids),
            patients_with_report=reports_by_lm[lm]["DFCI_MRN"].nunique(),
            patients_with_embedded_report=docs["DFCI_MRN"].nunique(), embedded_reports=len(docs),
            outlier_reports=int(docs["topic"].eq(-1).sum())))
        effects, curves = continuous_topic_association(docs, risk, lm, words)
        continuous_frames.append(effects)
        curve_frames.append(curves)
    continuous = pd.concat(continuous_frames, ignore_index=True).reindex(columns=CONTINUOUS_COLS)
    # One BH family per method over ALL tested topics and landmarks; failed/untestable rows remain NaN.
    for model, indices in continuous.groupby("model").groups.items():
        continuous.loc[indices, "q_value"] = bh_qvalues(continuous.loc[indices, "p_value"])
    continuous_curves = pd.concat(curve_frames, ignore_index=True).reindex(columns=CURVE_COLS)
    coverage = pd.DataFrame(continuous_coverage)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    continuous.to_csv(OUTPUT_DIR / "pathology_topic_continuous_association.csv", index=False)
    continuous_curves.to_csv(OUTPUT_DIR / "pathology_topic_risk_curves.csv", index=False)
    coverage.to_csv(OUTPUT_DIR / "pathology_topic_continuous_coverage.csv", index=False)
    display(coverage)
    for model in ("gee", "gee_adjusted", "patient_fraction_spearman"):
        sub = continuous[continuous["model"].eq(model)]
        print(f"\n{model}: {sub['status'].eq('ok').sum()} estimated; "
              f"{sub['q_value'].lt(TERM_FDR).sum()} at q < {TERM_FDR} across topics and landmarks")
        display(sub.sort_values("p_value").head(TERM_TOP_N).round(4).reset_index(drop=True))
    if not continuous_curves.empty:
        import matplotlib.pyplot as plt
        # Display the strongest unadjusted associations, including nonsignificant ones (q shown in each title).
        for _, row in continuous[continuous["model"].eq("gee") & continuous["status"].eq("ok")].nsmallest(6, "p_value").iterrows():
            curve = continuous_curves[continuous_curves["landmark"].eq(row["landmark"]) & continuous_curves["topic"].eq(row["topic"])]
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(curve["risk_score"], curve["probability"])
            ax.fill_between(curve["risk_score"], curve["ci95_low"], curve["ci95_high"], alpha=0.2)
            ax.set(xlabel="Patient risk score (continuous)", ylabel="Fitted probability of topic in a note", ylim=(0, 1),
                   title=f"Landmark {int(row['landmark'])} · topic {int(row['topic'])} · q={row['q_value']:.3g}")
            fig.tight_layout()
            plt.show()
            plt.close(fig)
